In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os


def load_results(kernel, strategy):
    save_dir = f"results/{strategy}/{kernel}"
    ys = []
    for filename in sorted(os.listdir(save_dir)):
        results = np.load(f"{save_dir}/{filename}")
        ys.append(results["ymin"])
    return np.array(ys)


plt.figure(figsize=(10, 5))
for kernel, strategy, color in [
    ("matern52", "lhs", "#1f77b4"),
    ("matern52", "bfgs", "#ff7f0e"),
    ("squaredexponential", "lhs", "#2ca02c"),
    ("squaredexponential", "bfgs", "#d62728"),
]:
    y = load_results(kernel, strategy)
    runs, max_acquisitions = y.shape
    x = np.arange(max_acquisitions)

    # ci on median
    qu = np.clip(0.5 + 1.96 * np.sqrt(0.25 / runs), 0, 1)
    ql = np.clip(0.5 - 1.96 * np.sqrt(0.25 / runs), 0, 1)
    ul = np.quantile(y, qu, axis=0)
    ll = np.quantile(y, ql, axis=0)

    plt.plot(x, np.median(y, axis=0), label=f"{kernel} + {strategy} ({runs} runs)", color=color)
    plt.fill_between(x, ll, ul, alpha=0.2, color=color)

plt.xlabel("Number of Acquisitions")
plt.ylabel("Minimum Function Value")
plt.title("Bayesian Optimization Results")
plt.legend()
plt.grid()
plt.show()